# Koddan Kasa · Levha 04 · Sürtünme: ağır küp daha geç kayar mı?

Videodaki deney: 8 cm'lik küp eğik bir rampanın ortasına duran hâlde konuyor. Rampa açısı 0,05° adımlarla taranıyor; 3 kütle (0,1 · 1 · 10 kg) × 3 sürtünme katsayısı (μ = 0,3 · 0,5 · 0,8).

Bizim sonuçlar (MuJoCo 3.17, varsayılan piramit sürtünme konisi):

| μ | teori arctan μ | 0,1 kg | 1 kg | 10 kg |
|---|---|---|---|---|
| 0,3 | 16,70° | 16,75° | 16,75° | 16,75° |
| 0,5 | 26,57° | 26,60° | 26,60° | 26,60° |
| 0,8 | 38,66° | 38,70° | 38,70° | 38,70° |

Kütle hem aşağı çeken kuvveti (m·g·sinθ) hem sürtünmenin sınırını (μ·m·g·cosθ) aynı oranda büyütüyor; eşitlikte sadeleşiyor: **tan θ = μ**.

## 1 · Kurulum

In [ ]:
!pip -q install mujoco==3.17.0

In [ ]:
import json, mujoco, numpy as np
print("mujoco", mujoco.__version__)

## 2 · Sahne: eğik rampa + küp

İki geom'un da `friction` değeri μ. MuJoCo iki geom'un sürtünmesini varsayılan olarak **büyük olanı** seçerek birleştirir, yani temasın μ'sü = μ.

In [ ]:
HALF = 0.04                    # küp yarı kenarı → 8 cm küp
RAMP = (0.40, 0.15, 0.01)      # rampa yarı boyutları → 80 × 30 × 2 cm

def xml(theta_deg, mu, mass, cone="pyramidal"):
    return f'''
<mujoco>
  <option cone="{cone}"/>
  <worldbody>
    <body name="ramp" euler="0 {-theta_deg} 0">
      <geom type="box" size="{RAMP[0]} {RAMP[1]} {RAMP[2]}" friction="{mu} 0.005 0.0001"/>
    </body>
    <body name="cube">
      <freejoint/>
      <geom type="box" size="{HALF} {HALF} {HALF}" mass="{mass}" friction="{mu} 0.005 0.0001"/>
    </body>
  </worldbody>
</mujoco>'''

## 3 · "Kayıyor" ne demek? (hız eşiği)

MuJoCo'nun teması **yumuşak**: eşiğin altında bile küp çok yavaş, sabit hızla "sızar" (creep; bizim ölçümde 0,1–1,4 cm/s). Bu yüzden yalnız "hız > X" demek yetmiyor.

Ölçüt: küp duran hâlde konur, 2 sn koşulur. `v1` = 0,5–1,0 sn ortalama aşağı hız, `v2` = 1,5–2,0 sn ortalama.
**Kayıyor ⇔ v2 > 2 cm/s VE v2 − v1 > 0,5 cm/s** (hem hızlı hem hızlanıyor). Gerçek kayma hızlanır: a = g(sinθ − μ cosθ).

In [ ]:
V_TH, A_TH = 2.0, 0.5   # cm/s

def speeds(theta_deg, mu, mass, cone="pyramidal"):
    m = mujoco.MjModel.from_xml_string(xml(theta_deg, mu, mass, cone))
    d = mujoco.MjData(m)
    mujoco.mj_forward(m, d)
    R = d.body("ramp").xmat.reshape(3, 3)
    n, s = R[:, 2], R[:, 0]                 # yüzey normali, rampa ekseni (yokuş yukarı)
    d.qpos[:3] = n * (RAMP[2] + HALF)       # küp yüzeye değiyor
    q = np.zeros(4); mujoco.mju_mat2Quat(q, R.flatten()); d.qpos[3:7] = q
    v = []
    while d.time < 2.0 - 1e-9:
        mujoco.mj_step(m, d)
        v.append(-d.qvel[:3] @ s * 100)     # aşağı hız, cm/s
    v, dt = np.array(v), m.opt.timestep
    return v[int(0.5/dt):int(1.0/dt)].mean(), v[int(1.5/dt):int(2.0/dt)].mean()

def slips(theta_deg, mu, mass, cone="pyramidal"):
    v1, v2 = speeds(theta_deg, mu, mass, cone)
    return v2 > V_TH and (v2 - v1) > A_TH

for th in (25.0, 26.5, 26.6, 27.0):
    v1, v2 = speeds(th, 0.5, 1.0)
    print(f"{th:5.2f}° · v1 {v1:5.2f} · v2 {v2:5.2f} cm/s · {'KAYIYOR' if slips(th, 0.5, 1.0) else 'duruyor (ya da yalnız sızıyor)'}")

## 4 · Tarama: 3 kütle × 3 μ, 0,05° adım (videodaki tablo)

In [ ]:
GRID = 0.05

def slip_angle(mu, mass, cone="pyramidal"):
    ref = np.degrees(np.arctan(mu))
    angles = np.round(np.arange(round((ref - 3) / GRID) * GRID, ref + 1 + 1e-9, GRID), 2)
    flags = [slips(float(a), mu, mass, cone) for a in angles]
    k = next(i for i in range(len(flags)) if all(flags[i:]))   # o ve üstündeki her açı kayıyor
    return float(angles[k])

result = {}
for mu in (0.3, 0.5, 0.8):
    row = [slip_angle(mu, m) for m in (0.1, 1.0, 10.0)]
    result[mu] = row
    print(f"μ {mu} · teori {np.degrees(np.arctan(mu)):5.2f}° · ölçülen (0,1 / 1 / 10 kg): {row}")

> Yerelde (MuJoCo 3.17) bu hücre videodaki sayıları birebir verdi: 16,75 · 26,60 · 38,70°, her satırda üç kütle aynı.
> Sapma teoriden en fazla +0,05°, yani ızgara adımı kadar.

## 5 · Tuzak: sürtünme konisi

MuJoCo varsayılan olarak sürtünme konisini bir **piramitle** yaklaşık alır (`cone="pyramidal"`). `elliptic` gerçek koniye daha yakın.
Biz küpü 45° döndürünce piramitte kayma açısı erken geldi (μ 0,8'de ~1° erken) ve geçiş düzensizleşti; eliptik konide düzgün. Videodaki koşular döndürülmemiş küp ve varsayılan piramit.

In [ ]:
for mu in (0.3, 0.5, 0.8):
    print(f"μ {mu} · piramit {slip_angle(mu, 1.0)}° · eliptik {slip_angle(mu, 1.0, 'elliptic')}° · teori {np.degrees(np.arctan(mu)):.2f}°")

## 6 · Kendin dene: tek sayı değiştir

Videodaki soru: **μ = 1 olsa kaç derecede kayar?** Önce tahmin et, sonra koştur. İstersen `MASS`'ı 1000 yap.

In [ ]:
MU, MASS = 1.0, 1.0      # ← değiştir
print(f"μ {MU}, {MASS} kg → kayma açısı {slip_angle(MU, MASS)}°  (teori arctan μ = {np.degrees(np.arctan(MU)):.2f}°)")

## 7 · Sonuç dosyası

Videodaki her sayı bu dosyadan (`results/04/result.json` düzeni, müfredat §4.3).

In [ ]:
import os
os.makedirs("results/04", exist_ok=True)
json.dump({"mujoco": mujoco.__version__, "criterion": "v2 > 2 cm/s ve v2 - v1 > 0,5 cm/s", "grid_deg": GRID,
           "slip_deg": {str(k): v for k, v in result.items()}}, open("results/04/result.json", "w"), ensure_ascii=False, indent=1)
print(open("results/04/result.json").read())